# AVIATION ACCIDENT ANALYSIS

## Overview




This project provides a data-driven analysis of aviation accidents to help stakeholders determine which aircraft are low risk and viable investment to purchase for commercial and private enterprises diversifying the companies portfilio by expanding to the aviation industry.

## 1. Business Understanding


The project results is to provide insights on the Aviation industry through it's functionality , operations and requirements for an entry level market entrance comprehension of risks and management structure.

### 1.1 Objectives

- Recommend Aircraft Make and Model viable for investment
- Find the frequency of accidents 
- Serverity of accident
- Determine Key factors affecting accident rates

## 2. Data understanding
Analysis sourced from [Aviation_Data](Dataset/Aviation_Data.csv) dataset provided by the National Transportation Saftey Board containing aviation accidents data from 1962-2023 in the United States and international waters.


In [491]:
# load the csv file into a dataframe  
import pandas as pd

df = pd.read_csv('Dataset/Aviation_Data.csv', index_col=1,low_memory=False)
df.head()

,Event.Id,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
Investigation.Type,,,,,,,,,,,,,,,,,,,,,
Accident,20001218X45444,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,Fatal(2),...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
Accident,20001218X45447,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,Fatal(4),...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
Accident,20061025X01555,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,Fatal(3),...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
Accident,20001218X45448,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,Fatal(2),...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
Accident,20041105X01764,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,Fatal(1),...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [492]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 90348 entries, Accident to Accident
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Accident.Number         88889 non-null  object 
 2   Event.Date              88889 non-null  object 
 3   Location                88837 non-null  object 
 4   Country                 88663 non-null  object 
 5   Latitude                34382 non-null  object 
 6   Longitude               34373 non-null  object 
 7   Airport.Code            50132 non-null  object 
 8   Airport.Name            52704 non-null  object 
 9   Injury.Severity         87889 non-null  object 
 10  Aircraft.damage         85695 non-null  object 
 11  Aircraft.Category       32287 non-null  object 
 12  Registration.Number     87507 non-null  object 
 13  Make                    88826 non-null  object 
 14  Model                   88797 non

In [493]:
df.isna().sum().sort_values(ascending=True)

Event.Id                   1459
Accident.Number            1459
Event.Date                 1459
Location                   1511
Make                       1522
Model                      1551
Amateur.Built              1561
Country                    1685
Injury.Severity            2459
Registration.Number        2841
Aircraft.damage            4653
Weather.Condition          5951
Total.Uninjured            7371
Number.of.Engines          7543
Purpose.of.flight          7651
Report.Status              7843
Engine.Type                8555
Total.Fatal.Injuries      12860
Total.Minor.Injuries      13392
Total.Serious.Injuries    13969
Publication.Date          16689
Broad.phase.of.flight     28624
Airport.Name              37644
Airport.Code              40216
Latitude                  55966
Longitude                 55975
Aircraft.Category         58061
FAR.Description           58325
Air.carrier               73700
Schedule                  77766
dtype: int64

The data has a lot of noise calling for cleaning and engineering of the 90348 columns and 30 rows to be ready and viable for analysis through:
- Dropping of irrelevant columns under the objectives threshold
- Grouping of columns
- Extraction of Months and Years from date and time columns
- Categorical splitting 
- Classification of Locations of accidents

## 3. Data Preperation



### 3.1 Libraries

In [ ]:
# Importing necessary liabraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pycountry_convert as pc # for converting country names to continent names
import pycountry               # for list of country names
import plotly.express as px    # for interactive visualizations plots of latitude and longitude







### 3.2 Dropping 

#### 3.2.1 Dropping columns

In [495]:
# dropping the columns that are not required for the analysis
df = df.drop(columns=['Event.Id','Accident.Number','Airport.Code','Airport.Name','Registration.Number','FAR.Description','Air.carrier','Report.Status','Publication.Date'])


#### 3.2.2 Dropping Missing values

In [496]:
# drop missing data in the subsets
df = df.dropna(subset=['Event.Date','Location', 'Country','Injury.Severity','Make','Model','Purpose.of.flight'])



#### 3.3 Accidents per period

In [498]:
# Event.Date conversion to datetime formart
df['Event.Date'] = pd.to_datetime(df['Event.Date'])

# extract 'year' in datetime
df['year'] = df['Event.Date'].dt.year

#group every 10years from 1962-2023
bins=np.arange(1962, 2035, 10)
labels = [f'{i}-{i+9}' for i in bins[:-1]]
df['year_group'] = pd.cut(df['year'], bins=bins, labels=labels, right=False)
df=df.dropna(subset='year_group')

# Season extraction from the date
def get_season(Date):
    month = Date.month
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'
    
df['season'] = df['Event.Date'].apply(get_season)    

|Years|Number of Accidents|
|:---|---:|
|1962-1971|1|
|1972-1981|4|
|1982-1991|29447|
|1992-2001|21800|
|2002-2011|16988|
|2012-2021|12889|
|2022 & 2023|1198|


#### 3.4 Locations of Accidents

In [499]:
# accidents location on continent level

# initializing the 'State' column with None values
US_State = None
# creating a boolean mask to identify rows where 'Country' is 'United States'
mask = df['Country'] == 'United States'
df.loc[mask, 'US_State'] = df.loc[mask, 'Location'].str.extract(r',\s*([^,]+)$')[0]

# united states regions of state
northeast = ['ME','NH','VT','MA','RI','CT','NY','NJ','PA']
southeast = ['DE','MD','DC','VA','WV','NC','SC','GA','FL','KY','TN','MS','AL','OK','AR','LA']
midwest = ['OH','MI','IN','IL','WI','MN','IA','MO','ND','SD','NE','KS']
southwest = ['TX','NM','AZ']
west = ['CO','WY','MT','ID','WA','OR','UT','NV','CA','AK','HI']
atlantic_continents = ['North America', 'Europe', 'ATLANTIC OCEAN', 'GULF OF MEXICO']
pacific_continents = ['Asia', 'South America', 'Pacific Ocean']
indian_continents = ['Africa']
southern_continents = ['Australia', 'Antarctica', 'Oceania']

# function to assign region based on state
def assign_region(US_State):
    if US_State in northeast:
        return 'Northeast USA'
    elif US_State in southeast:
        return 'Southeast USA'
    elif US_State in midwest:
        return 'Midwest USA'
    elif US_State in southwest:
        return 'Southwest USA'
    elif US_State in west:
        return 'West USA'
    else:
        return None

def country_to_continent(country):
  try:
        country_code = pc.country_name_to_country_alpha2(country)
        continent_code = pc.country_alpha2_to_continent_code(country_code)
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        return continent_name
  except:
    return None
    
df['State_Region'] = df.apply(lambda row: assign_region(row['US_State']) if row['Country'] == 'United States' else country_to_continent(row['Country']), axis=1) 


def assign_ocean(country):
    if country in atlantic_continents:
        return 'Atlantic Ocean'
    elif country in pacific_continents:
        return 'Pacific Ocean'
    elif country  in indian_continents:
        return 'Indian Ocean'
    elif country in southern_continents:
        return 'Southern Ocean'
    else:
        return country
    
df['State_Region'] = df['State_Region'].apply(assign_ocean)



|State_Region|Number of Accidents|
|:---|---:|
|West USA|26947|
|Southeast USA|20333|
|Midwest USA|15054|
|Southwest USA|9882|
|Northeast USA|7362|
|Atlantic Ocean|1419|
|Pacific Ocean|651|
|Southern Ocean|135|
|Indian Ocean|109|

In [502]:
df['Broad.phase.of.flight'].value_counts()

Broad.phase.of.flight
Landing        15213
Takeoff        12247
Cruise         10007
Maneuvering     8072
Approach        6390
Climb           1957
Descent         1822
Taxi            1813
Go-around       1343
Standing         851
Unknown          532
Other            116
Name: count, dtype: int64

In [503]:
df['Broad.phase.of.flight'].dropna(inplace=True)

In [504]:
df.head()

,Event.Date,Location,Country,Latitude,Longitude,Injury.Severity,Aircraft.damage,Aircraft.Category,Make,Model,...,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,year,year_group,season,US_State,State_Region
Investigation.Type,,,,,,,,,,,,,,,,,,,,,
Accident,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,Fatal(4),Destroyed,NaN,Piper,PA24-180,...,0.0,0.0,0.0,UNK,Unknown,1962,1962-1971,Summer,CA,West USA
Accident,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,Fatal(3),Destroyed,NaN,Cessna,172M,...,NaN,NaN,NaN,IMC,Cruise,1974,1972-1981,Summer,VA,Southeast USA
Accident,1977-06-19,"EUREKA, CA",United States,NaN,NaN,Fatal(2),Destroyed,NaN,Rockwell,112,...,0.0,0.0,0.0,IMC,Cruise,1977,1972-1981,Summer,CA,West USA
Accident,1979-08-02,"Canton, OH",United States,NaN,NaN,Fatal(1),Destroyed,NaN,Cessna,501,...,2.0,NaN,0.0,VMC,Approach,1979,1972-1981,Summer,OH,Midwest USA
Accident,1981-08-01,"COTTON, MN",United States,NaN,NaN,Fatal(4),Destroyed,NaN,Cessna,180,...,0.0,0.0,0.0,IMC,Unknown,1981,1972-1981,Summer,MN,Midwest USA


## Exploratory Data Analysis
  
   

In [ ]:
# accidents per season and weather condition
df.groupby(['season', 'Weather.Condition']).size().unstack(fill_value=0)

- Level of aircraft damage

The stakeholders are looking to invest in purchase of commercial and private eneterprises planes .This is best represented by 
  - 'Personal'
  - 'Business'
  - 'Executive'


in Purpose.of.flight
Create a pivot_table to display number of aircrafts to be invested in damage levels.

In [507]:
x=['Personal','Business','Executive/corporate']
flight_purpose=df[df['Purpose.of.flight'].isin(x)]

craft_damage=flight_purpose.pivot_table(index='Aircraft.damage', columns='Purpose.of.flight',aggfunc='size')
print(craft_damage)

Purpose.of.flight  Business  Executive/corporate  Personal
Aircraft.damage                                           
Destroyed              1177                  159     10519
Minor                   106                   33       495
Substantial            2623                  327     37941
Unknown                   3                    2        32


- Serverty of accidents

In [508]:
df['total_injuries']=df[['Total.Fatal.Injuries','Total.Minor.Injuries','Total.Serious.Injuries']].fillna(0).sum(axis=1).astype(int)
df.head()  

,Event.Date,Location,Country,Latitude,Longitude,Injury.Severity,Aircraft.damage,Aircraft.Category,Make,Model,...,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,year,year_group,season,US_State,State_Region,total_injuries
Investigation.Type,,,,,,,,,,,,,,,,,,,,,
Accident,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,Fatal(4),Destroyed,NaN,Piper,PA24-180,...,0.0,0.0,UNK,Unknown,1962,1962-1971,Summer,CA,West USA,4
Accident,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,Fatal(3),Destroyed,NaN,Cessna,172M,...,NaN,NaN,IMC,Cruise,1974,1972-1981,Summer,VA,Southeast USA,3
Accident,1977-06-19,"EUREKA, CA",United States,NaN,NaN,Fatal(2),Destroyed,NaN,Rockwell,112,...,0.0,0.0,IMC,Cruise,1977,1972-1981,Summer,CA,West USA,2
Accident,1979-08-02,"Canton, OH",United States,NaN,NaN,Fatal(1),Destroyed,NaN,Cessna,501,...,NaN,0.0,VMC,Approach,1979,1972-1981,Summer,OH,Midwest USA,3
Accident,1981-08-01,"COTTON, MN",United States,NaN,NaN,Fatal(4),Destroyed,NaN,Cessna,180,...,0.0,0.0,IMC,Unknown,1981,1972-1981,Summer,MN,Midwest USA,4


In [ ]:
fig = px.scatter_geo(df, lat = 'Latitude', lon = 'Longitude', hover_name = 'Location',
                     color='Weather.Condition', title='Global Aviation Accidents', projection = 'kavrayskiy7')
fig.show()

- Least accident involved make

Least make and model of aircraft determined by the number of accidents incurred and total number of injuries on the accident.

In [509]:

# 1. Filter for x purposes and minor aircraft damage
x = ['Personal', 'Business', 'Executive']
df_filtered = df[
    df['Purpose.of.flight'].isin(x) &
    (df['Aircraft.damage'].str.lower() == 'minor')
]

# 2. Group by Make and Model
accident_summary = (
    df_filtered.groupby(['Make', 'Model'])
      .agg(
          Accident_Count=('Model', 'size'),
          total_injuries=('total_injuries', 'sum')
      )
      .reset_index()
)

# 3. Filter for accident and  injuries
filtered = accident_summary[
    (accident_summary['Accident_Count'] == 1) &
    (accident_summary['total_injuries'] == 0)
]

# 4. Random pick 
sample_size = min(6, len(filtered))
random_samples = filtered.sample(n=sample_size, random_state=42).reset_index(drop=True)

print('Least involved make in accidents')
print(random_samples)

Least involved make in accidents
                       Make       Model  Accident_Count  total_injuries
0                    CESSNA       172RG               1               0
1  New Piper Aircraft, Inc.  PA-46-350P               1               0
2        CIRRUS DESIGN CORP        SR20               1               0
3             STARDUSTER II      SA-300               1               0
4                    Cessna        310R               1               0
5                    Cessna   550 Bravo               1               0


## Conclusion

Through filtering and EDA analysis of the dataset the planes of make :Make CESSNA , New Piper Aircraft, Inc. ,COLUMBIA , STARDUSTER II , Cessna ,Cessna  and model:Model 421, PA-46-350P, LC41, SA-300, 337DA150L  respectively had the least number of accidents in relevance to total injuries,investors purpose of crafts and severity of damage of craft crashed.     

## Recommendation  

Would definetly recommend purchase of 
- New Piper Aircraft,Inc.  model:PA-46-350P
- COLUMBIA.                model:LC41
- STARDUSTER II.           model:SA-300


.For startup venture into the aviation industry.
